# 🧠 AI Intelligence Collector for NotebookLM

**Mission:** Collect the latest important AI content from YouTube and Twitter, format it beautifully for NotebookLM.

**Philosophy:** We don't analyze. We curate. NotebookLM does the deep thinking.

---

## What This Does

1. **YouTube Intelligence**: Finds important AI videos, extracts metadata, prepares for NotebookLM
2. **Twitter Intelligence**: Captures AI discourse threads, formats as beautiful markdown
3. **Export Package**: Creates NotebookLM-ready sources you can upload instantly

## Workflow

```
Run Notebook → Get Curated Sources → Upload to NotebookLM → Audio Overview Magic ✨
```

## 📦 Setup & Installation

In [ ]:
# Install required packages
!pip install -q youtube-transcript-api google-api-python-client tweepy python-dotenv anthropic requests beautifulsoup4 markdown pyyaml

In [ ]:
import os
import json
import re
from datetime import datetime, timedelta
from typing import List, Dict, Optional
from pathlib import Path
import requests
from dataclasses import dataclass, asdict
from enum import Enum

# YouTube & Transcript
from youtube_transcript_api import YouTubeTranscriptApi
from googleapiclient.discovery import build

# Twitter (now X)
import tweepy

# AI Understanding
import anthropic

# Environment
from dotenv import load_dotenv
load_dotenv()

print("✅ All packages loaded")

## 🔑 API Configuration

Create a `.env` file in the project root with:

```env
# YouTube Data API v3
YOUTUBE_API_KEY=your_youtube_api_key

# Twitter/X API (v2)
TWITTER_BEARER_TOKEN=your_twitter_bearer_token

# Anthropic Claude API (for intelligence ranking)
ANTHROPIC_API_KEY=your_anthropic_api_key
```

**Get Your Keys:**
- YouTube: https://console.cloud.google.com/apis/credentials
- Twitter: https://developer.twitter.com/en/portal/dashboard
- Anthropic: https://console.anthropic.com/

In [ ]:
# API Keys
YOUTUBE_API_KEY = os.getenv('YOUTUBE_API_KEY')
TWITTER_BEARER_TOKEN = os.getenv('TWITTER_BEARER_TOKEN')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

# Initialize clients
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY) if YOUTUBE_API_KEY else None
twitter_client = tweepy.Client(bearer_token=TWITTER_BEARER_TOKEN) if TWITTER_BEARER_TOKEN else None
claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print("🔑 API Clients initialized")
print(f"   YouTube: {'✅' if youtube else '❌ (set YOUTUBE_API_KEY)'}")
print(f"   Twitter: {'✅' if twitter_client else '❌ (set TWITTER_BEARER_TOKEN)'}")
print(f"   Claude:  {'✅' if claude_client else '❌ (set ANTHROPIC_API_KEY)'}")

## 🎯 Configuration: What To Track

Define your intelligence sources here.

In [ ]:
# YouTube Channels to Monitor (AI thought leaders + VCs)
YOUTUBE_CHANNELS = [
    'UCbfYPyITQ-7l4upoX8nvctg',  # Two Minute Papers
    'UCUHW94eEFW7hkUMVaZz4eDg',  # Yannic Kilcher
    'UC5vz0bX7eWePpuKb7DqRYsQ',  # AI Explained
    # Add your favorite AI channels
]

# YouTube Search Terms (including channel names for handle-based search)
YOUTUBE_SEARCH_TERMS = [
    'GPT-5',
    'Claude 3.5',
    'o3 OpenAI',
    'Gemini 2.0',
    'OpenAI news',
    'AI breakthrough 2025',
    'LLM research',
    'Anthropic',
    'Sequoia Capital AI',        # Sequoia VC insights
    'a16z AI',                    # Andreessen Horowitz AI
    'Sequoia AI investing',
    'a16z artificial intelligence',
]

# Twitter Accounts to Monitor (AI leaders + VCs)
TWITTER_ACCOUNTS = [
    # Companies
    'AnthropicAI',
    'OpenAI',
    'GoogleDeepMind',
    
    # Researchers & Thought Leaders
    'karpathy',              # Andrej Karpathy
    'ylecun',                # Yann LeCun
    'sama',                  # Sam Altman
    'DrJimFan',              # Jim Fan
    'goodfellow_ian',        # Ian Goodfellow
    
    # Venture Capital (following the smart money)
    'sequoia',               # Sequoia Capital
    'a16z',                  # Andreessen Horowitz
    'a16zcrypto',            # a16z Crypto
    'pmarca',                # Marc Andreessen (a16z co-founder)
    'cdixon',                # Chris Dixon (a16z partner)
    
    # Add more
]

# Twitter Hashtags
TWITTER_HASHTAGS = [
    '#AI',
    '#LLM',
    '#MachineLearning',
    '#AIResearch',
    '#GenerativeAI',
]

# Time window (how far back to look)
DAYS_TO_LOOK_BACK = 7

print(f"📊 Monitoring {len(YOUTUBE_CHANNELS)} YouTube channels")
print(f"📊 Searching {len(YOUTUBE_SEARCH_TERMS)} YouTube terms (including VC firms)")
print(f"📊 Monitoring {len(TWITTER_ACCOUNTS)} Twitter accounts (including Sequoia & a16z)")
print(f"📊 Looking back {DAYS_TO_LOOK_BACK} days")

## 📊 Data Models

In [ ]:
@dataclass
class YouTubeVideo:
    """Represents a YouTube video with intelligence metadata"""
    video_id: str
    url: str
    title: str
    channel: str
    published_at: str
    view_count: int
    like_count: int
    duration: str
    description: str
    has_transcript: bool
    importance_score: Optional[float] = None
    ai_summary: Optional[str] = None
    key_topics: Optional[List[str]] = None
    
    def to_markdown(self) -> str:
        """Format as NotebookLM-ready markdown"""
        md = f"""# {self.title}

**Channel:** {self.channel}  
**Published:** {self.published_at}  
**Views:** {self.view_count:,} | **Likes:** {self.like_count:,}  
**Duration:** {self.duration}  
**URL:** {self.url}

## Description

{self.description}
"""
        if self.ai_summary:
            md += f"\n## AI Summary\n\n{self.ai_summary}\n"
        
        if self.key_topics:
            md += f"\n## Key Topics\n\n{', '.join(self.key_topics)}\n"
        
        if self.importance_score:
            md += f"\n## Importance Score: {self.importance_score}/10\n"
        
        return md


@dataclass
class TwitterThread:
    """Represents a Twitter thread"""
    thread_id: str
    author: str
    author_handle: str
    tweets: List[Dict]
    created_at: str
    total_likes: int
    total_retweets: int
    importance_score: Optional[float] = None
    ai_summary: Optional[str] = None
    
    def to_markdown(self) -> str:
        """Format as NotebookLM-ready markdown"""
        md = f"""# Twitter Thread by @{self.author_handle}

**Author:** {self.author} (@{self.author_handle})  
**Date:** {self.created_at}  
**Engagement:** {self.total_likes:,} likes | {self.total_retweets:,} retweets

---

"""
        for i, tweet in enumerate(self.tweets, 1):
            md += f"**Tweet {i}:**\n\n{tweet['text']}\n\n"
        
        if self.ai_summary:
            md += f"---\n\n## AI Summary\n\n{self.ai_summary}\n"
        
        if self.importance_score:
            md += f"\n## Importance Score: {self.importance_score}/10\n"
        
        return md


print("✅ Data models defined")

## 🎬 YouTube Intelligence Collector

In [ ]:
def extract_video_id(url: str) -> Optional[str]:
    """Extract video ID from various YouTube URL formats"""
    patterns = [
        r'(?:youtube\.com\/watch\?v=|youtu\.be\/)([^&\n?#]+)',
        r'youtube\.com\/embed\/([^&\n?#]+)',
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    return url if len(url) == 11 else None


def get_video_details(video_id: str) -> Optional[YouTubeVideo]:
    """Fetch detailed information about a YouTube video"""
    if not youtube:
        print("❌ YouTube API not configured")
        return None
    
    try:
        response = youtube.videos().list(
            part='snippet,statistics,contentDetails',
            id=video_id
        ).execute()
        
        if not response['items']:
            return None
        
        video = response['items'][0]
        snippet = video['snippet']
        stats = video['statistics']
        
        # Check if transcript available
        has_transcript = False
        try:
            YouTubeTranscriptApi.get_transcript(video_id)
            has_transcript = True
        except:
            pass
        
        return YouTubeVideo(
            video_id=video_id,
            url=f"https://www.youtube.com/watch?v={video_id}",
            title=snippet['title'],
            channel=snippet['channelTitle'],
            published_at=snippet['publishedAt'],
            view_count=int(stats.get('viewCount', 0)),
            like_count=int(stats.get('likeCount', 0)),
            duration=video['contentDetails']['duration'],
            description=snippet['description'],
            has_transcript=has_transcript,
        )
    except Exception as e:
        print(f"❌ Error fetching video {video_id}: {e}")
        return None


def search_youtube_videos(query: str, max_results: int = 10) -> List[YouTubeVideo]:
    """Search YouTube for videos matching query"""
    if not youtube:
        return []
    
    try:
        # Calculate date threshold
        published_after = (datetime.now() - timedelta(days=DAYS_TO_LOOK_BACK)).isoformat() + 'Z'
        
        response = youtube.search().list(
            part='id',
            q=query,
            type='video',
            publishedAfter=published_after,
            maxResults=max_results,
            order='relevance',
            relevanceLanguage='en',
        ).execute()
        
        videos = []
        for item in response.get('items', []):
            video_id = item['id']['videoId']
            video = get_video_details(video_id)
            if video:
                videos.append(video)
        
        return videos
    except Exception as e:
        print(f"❌ Error searching YouTube for '{query}': {e}")
        return []


def get_channel_latest_videos(channel_id: str, max_results: int = 10) -> List[YouTubeVideo]:
    """Get latest videos from a channel"""
    if not youtube:
        return []
    
    try:
        published_after = (datetime.now() - timedelta(days=DAYS_TO_LOOK_BACK)).isoformat() + 'Z'
        
        response = youtube.search().list(
            part='id',
            channelId=channel_id,
            type='video',
            publishedAfter=published_after,
            maxResults=max_results,
            order='date',
        ).execute()
        
        videos = []
        for item in response.get('items', []):
            video_id = item['id']['videoId']
            video = get_video_details(video_id)
            if video:
                videos.append(video)
        
        return videos
    except Exception as e:
        print(f"❌ Error fetching channel {channel_id}: {e}")
        return []


print("✅ YouTube collector ready")

## 🐦 Twitter Intelligence Collector

In [ ]:
def get_user_recent_tweets(username: str, max_results: int = 10) -> List[Dict]:
    """Get recent tweets from a user"""
    if not twitter_client:
        return []
    
    try:
        # Get user ID
        user = twitter_client.get_user(username=username)
        if not user.data:
            return []
        
        user_id = user.data.id
        
        # Calculate time range
        start_time = datetime.now() - timedelta(days=DAYS_TO_LOOK_BACK)
        
        # Get tweets
        tweets = twitter_client.get_users_tweets(
            id=user_id,
            max_results=max_results,
            start_time=start_time,
            tweet_fields=['created_at', 'public_metrics', 'conversation_id'],
            exclude=['retweets', 'replies'],
        )
        
        if not tweets.data:
            return []
        
        return [
            {
                'id': tweet.id,
                'text': tweet.text,
                'created_at': tweet.created_at.isoformat() if tweet.created_at else None,
                'likes': tweet.public_metrics['like_count'],
                'retweets': tweet.public_metrics['retweet_count'],
            }
            for tweet in tweets.data
        ]
    except Exception as e:
        print(f"❌ Error fetching tweets from @{username}: {e}")
        return []


def search_twitter(query: str, max_results: int = 10) -> List[Dict]:
    """Search Twitter for recent tweets"""
    if not twitter_client:
        return []
    
    try:
        start_time = datetime.now() - timedelta(days=DAYS_TO_LOOK_BACK)
        
        tweets = twitter_client.search_recent_tweets(
            query=query,
            max_results=max_results,
            start_time=start_time,
            tweet_fields=['created_at', 'public_metrics', 'author_id'],
        )
        
        if not tweets.data:
            return []
        
        return [
            {
                'id': tweet.id,
                'text': tweet.text,
                'created_at': tweet.created_at.isoformat() if tweet.created_at else None,
                'likes': tweet.public_metrics['like_count'],
                'retweets': tweet.public_metrics['retweet_count'],
                'author_id': tweet.author_id,
            }
            for tweet in tweets.data
        ]
    except Exception as e:
        print(f"❌ Error searching Twitter for '{query}': {e}")
        return []


print("✅ Twitter collector ready")

## 🤖 AI Intelligence Layer (Claude)

Uses Claude to understand and rank content importance.

In [ ]:
def analyze_video_importance(video: YouTubeVideo) -> YouTubeVideo:
    """Use Claude to analyze video importance and extract insights"""
    if not claude_client:
        return video
    
    try:
        prompt = f"""Analyze this YouTube video about AI and provide:
1. Importance score (1-10) - how significant is this for someone tracking AI developments?
2. Brief summary (2-3 sentences)
3. Key topics (3-5 keywords)

Title: {video.title}
Channel: {video.channel}
Description: {video.description}
Views: {video.view_count:,}
Likes: {video.like_count:,}

Respond in JSON format:
{{
  "importance_score": 8.5,
  "summary": "...",
  "key_topics": ["topic1", "topic2", "topic3"]
}}"""
        
        response = claude_client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}]
        )
        
        # Parse response
        content = response.content[0].text
        # Extract JSON from potential markdown code blocks
        json_match = re.search(r'\{[\s\S]*\}', content)
        if json_match:
            analysis = json.loads(json_match.group())
            video.importance_score = analysis.get('importance_score')
            video.ai_summary = analysis.get('summary')
            video.key_topics = analysis.get('key_topics', [])
    except Exception as e:
        print(f"⚠️  Could not analyze video {video.video_id}: {e}")
    
    return video


def analyze_tweets_importance(tweets: List[Dict]) -> tuple[float, str]:
    """Use Claude to analyze a set of tweets"""
    if not claude_client or not tweets:
        return None, None
    
    try:
        tweets_text = "\n\n".join([f"Tweet {i+1}: {t['text']}" for i, t in enumerate(tweets)])
        
        prompt = f"""Analyze these tweets about AI and provide:
1. Importance score (1-10) - how significant is this content?
2. Brief summary of the main insights

{tweets_text}

Respond in JSON format:
{{
  "importance_score": 7.5,
  "summary": "..."
}}"""
        
        response = claude_client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=512,
            messages=[{"role": "user", "content": prompt}]
        )
        
        content = response.content[0].text
        json_match = re.search(r'\{[\s\S]*\}', content)
        if json_match:
            analysis = json.loads(json_match.group())
            return analysis.get('importance_score'), analysis.get('summary')
    except Exception as e:
        print(f"⚠️  Could not analyze tweets: {e}")
    
    return None, None


print("✅ AI intelligence layer ready")

## 🎯 Run Collection

Execute the intelligence gathering operation.

In [ ]:
print("🚀 Starting AI Intelligence Collection...\n")

all_videos = []
all_tweets = []

# Collect from YouTube channels
if youtube and YOUTUBE_CHANNELS:
    print("📺 Collecting from YouTube channels...")
    for channel_id in YOUTUBE_CHANNELS:
        print(f"   Fetching {channel_id}...")
        videos = get_channel_latest_videos(channel_id, max_results=5)
        all_videos.extend(videos)
        print(f"   ✅ Found {len(videos)} videos")
    print()

# Search YouTube
if youtube and YOUTUBE_SEARCH_TERMS:
    print("🔍 Searching YouTube...")
    for term in YOUTUBE_SEARCH_TERMS:
        print(f"   Searching: {term}")
        videos = search_youtube_videos(term, max_results=5)
        all_videos.extend(videos)
        print(f"   ✅ Found {len(videos)} videos")
    print()

# Collect from Twitter
if twitter_client and TWITTER_ACCOUNTS:
    print("🐦 Collecting from Twitter accounts...")
    for username in TWITTER_ACCOUNTS:
        print(f"   Fetching @{username}...")
        tweets = get_user_recent_tweets(username, max_results=5)
        all_tweets.extend(tweets)
        print(f"   ✅ Found {len(tweets)} tweets")
    print()

# Deduplicate videos
seen_video_ids = set()
unique_videos = []
for video in all_videos:
    if video.video_id not in seen_video_ids:
        seen_video_ids.add(video.video_id)
        unique_videos.append(video)

print(f"\n📊 Collection Summary:")
print(f"   YouTube Videos: {len(unique_videos)}")
print(f"   Twitter Posts: {len(all_tweets)}")
print(f"\n✅ Collection complete!")

## 🧠 AI Analysis & Ranking

Let Claude analyze and rank the content by importance.

In [ ]:
if claude_client and unique_videos:
    print("🤖 Analyzing videos with Claude...\n")
    analyzed_videos = []
    
    for i, video in enumerate(unique_videos[:20], 1):  # Limit to first 20 to save API calls
        print(f"   [{i}/{min(20, len(unique_videos))}] Analyzing: {video.title[:50]}...")
        analyzed_video = analyze_video_importance(video)
        analyzed_videos.append(analyzed_video)
    
    # Sort by importance
    analyzed_videos.sort(key=lambda v: v.importance_score or 0, reverse=True)
    unique_videos = analyzed_videos
    
    print(f"\n✅ Analysis complete!")
    print(f"\n🏆 Top 5 Most Important Videos:")
    for i, video in enumerate(unique_videos[:5], 1):
        score = video.importance_score or 0
        print(f"   {i}. [{score:.1f}/10] {video.title}")
else:
    print("⚠️  Skipping AI analysis (Claude API not configured or no videos)")

## 📦 Export for NotebookLM

Create beautifully formatted sources ready to upload to NotebookLM.

In [ ]:
# Create export directory
export_dir = Path('notebooklm_sources')
export_dir.mkdir(exist_ok=True)

# Add timestamp to this batch
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
batch_dir = export_dir / f'batch_{timestamp}'
batch_dir.mkdir(exist_ok=True)

print(f"📦 Exporting to: {batch_dir}\n")

# Export YouTube videos as markdown
if unique_videos:
    videos_dir = batch_dir / 'youtube_videos'
    videos_dir.mkdir(exist_ok=True)
    
    # Create individual markdown files for each video
    for video in unique_videos:
        safe_title = re.sub(r'[^\w\s-]', '', video.title)[:50]
        filename = f"{safe_title}.md"
        filepath = videos_dir / filename
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(video.to_markdown())
    
    # Create a master YouTube URLs file (NotebookLM can ingest these directly!)
    urls_file = batch_dir / 'youtube_urls.txt'
    with open(urls_file, 'w') as f:
        f.write("# Important AI Videos (Last 7 Days)\n\n")
        for video in unique_videos:
            f.write(f"{video.url}\n")
    
    print(f"✅ Exported {len(unique_videos)} YouTube videos")
    print(f"   - Individual markdown files: {videos_dir}")
    print(f"   - URL list: {urls_file}")

# Export Twitter content as markdown
if all_tweets:
    twitter_dir = batch_dir / 'twitter_content'
    twitter_dir.mkdir(exist_ok=True)
    
    # Group tweets by date
    tweets_by_date = {}
    for tweet in all_tweets:
        date = tweet['created_at'][:10] if tweet['created_at'] else 'unknown'
        if date not in tweets_by_date:
            tweets_by_date[date] = []
        tweets_by_date[date].append(tweet)
    
    # Create daily summaries
    for date, tweets in tweets_by_date.items():
        filepath = twitter_dir / f'tweets_{date}.md'
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(f"# AI Twitter Discourse - {date}\n\n")
            for tweet in tweets:
                f.write(f"## Tweet\n\n")
                f.write(f"{tweet['text']}\n\n")
                f.write(f"**Engagement:** {tweet['likes']} likes | {tweet['retweets']} retweets\n\n")
                f.write("---\n\n")
    
    print(f"✅ Exported {len(all_tweets)} tweets")
    print(f"   - Daily summaries: {twitter_dir}")

# Create master index
index_file = batch_dir / 'README.md'
with open(index_file, 'w') as f:
    f.write(f"""# AI Intelligence Report

**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Time Period:** Last {DAYS_TO_LOOK_BACK} days

## Summary

- **YouTube Videos:** {len(unique_videos)}
- **Twitter Posts:** {len(all_tweets)}

## How to Use with NotebookLM

1. Go to [NotebookLM](https://notebooklm.google.com)
2. Create a new notebook
3. Upload sources:
   - Upload `youtube_urls.txt` directly (NotebookLM will fetch video transcripts)
   - Upload markdown files from `youtube_videos/` for detailed context
   - Upload markdown files from `twitter_content/` for Twitter discourse
4. Ask NotebookLM to create an Audio Overview
5. Listen to your personalized AI news podcast! 🎧

## Top Videos

""")
    
    for i, video in enumerate(unique_videos[:10], 1):
        score = f" [{video.importance_score:.1f}/10]" if video.importance_score else ""
        f.write(f"{i}.{score} [{video.title}]({video.url})\n")
        if video.ai_summary:
            f.write(f"   - {video.ai_summary}\n")

print(f"\n✅ Master index: {index_file}")
print(f"\n🎉 Export complete! Ready for NotebookLM.")
print(f"\n📂 Open folder: {batch_dir}")

## 🎧 Next Steps

1. **Open the export folder** (shown above)
2. **Go to [NotebookLM](https://notebooklm.google.com)**
3. **Create a new notebook** titled "AI Intelligence - [Today's Date]"
4. **Upload sources:**
   - Start with `youtube_urls.txt` (easiest - NotebookLM handles everything)
   - Add markdown files for deeper context
5. **Generate Audio Overview** - Let NotebookLM create your AI news podcast
6. **Ask questions** like:
   - "What are the most important developments this week?"
   - "Are there any emerging trends?"
   - "What should I pay attention to?"

---

## 🔄 Automation Ideas

Run this notebook:
- **Daily:** Morning intelligence brief
- **Weekly:** Deep dive on trends
- **On-demand:** When major AI news drops

**Pro Tip:** Use GitHub Actions or cron to run this automatically and get your sources ready every morning!

## 🎨 Customization

Scroll back up and modify:
- `YOUTUBE_CHANNELS` - Add your favorite AI channels
- `YOUTUBE_SEARCH_TERMS` - Add specific topics you're tracking
- `TWITTER_ACCOUNTS` - Follow the AI voices you trust
- `DAYS_TO_LOOK_BACK` - Adjust time window

Then re-run all cells!